# DEST - CIFAR100 Pareado (seeds 52-61, 5 metodos)
Diseno **pareado**: para cada semilla se corren los 5 samplers seguidos.
Con la misma seed: mismos pesos iniciales, mismo dropout determinista y mismo split
(`split_seed=0`); **lo unico distinto entre metodos es el orden del sampler**.

| Bloque | Contenido | Runs |
|---|---|---|
| 1 | seeds 52-61 x 5 metodos | 50 |
| 2 | collatz_v2 seeds 42-51 (catch-up) | 10 |

**Reanudable:** si se cae Colab, Celda 1 + Celda 2 de nuevo; lo guardado se salta.
Puedes detenerte al terminar cualquier bloque de semilla: cada uno cierra un set pareado completo.
Al final la Celda 2 imprime el analisis pareado (diff por seed, t pareado, victorias).


In [ ]:
import os, sys, json, zipfile, torch, subprocess, importlib

try:
    import seaborn, sklearn, tqdm
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           'seaborn', 'scikit-learn', 'tqdm', 'matplotlib', '-q'])

lib_files = {
    "__init__.py": "# dest_lib package\n",
    "config.py": "import os\nfrom dataclasses import dataclass, field\nfrom typing import List, Optional, Dict, Any\n\n@dataclass\nclass RunResult:\n    experiment_id: str\n    dataset: str\n    sampler_name: str\n    seed: int\n    mode: str\n    train_losses: List[float]\n    val_losses: List[float]\n    test_losses: List[float]\n    train_accs: List[float]\n    val_accs: List[float]\n    test_accs: List[float]\n    generalization_gaps: List[float]\n    f1_per_epoch: List[float]\n    precision_per_epoch: List[float]\n    recall_per_epoch: List[float]\n    final_test_acc: float\n    final_test_loss: float\n    final_f1: float\n    final_precision: float\n    final_recall: float\n    final_ece: float\n    final_generalization_gap: float\n    convergence_epoch_90: Optional[int]\n    convergence_epoch_95: Optional[int]\n    best_test_acc: float\n    best_test_epoch: int\n    sampler_time_per_epoch: List[float]\n    train_time_per_epoch: List[float]\n    eval_time_per_epoch: List[float]\n    total_time_per_epoch: List[float]\n    total_runtime_seconds: float\n    samples_per_second: List[float]\n    gpu_memory_peak_mb: float\n    train_loss_variance: float\n    test_acc_variance: float\n    config_snapshot: Dict[str, Any]\n    timestamp: str\n    status: str = \"COMPLETE\"\n\n\n# Canonical dataset -> model mapping\nDATASET_MODEL_MAP: Dict[str, str] = {\n    \"MNIST\":        \"SmallCNN\",\n    \"FASHIONMNIST\": \"SmallCNN\",\n    \"CIFAR10\":      \"ResNet9\",\n    \"CIFAR100\":     \"ResNet18\",\n    \"TINYIMAGENET\": \"ResNet18\",\n}\n\n# Dataset difficulty rank (for scaling plot x-axis)\nDATASET_DIFFICULTY: Dict[str, int] = {\n    \"MNIST\":        0,\n    \"FASHIONMNIST\": 1,\n    \"CIFAR10\":      2,\n    \"CIFAR100\":     3,\n    \"TINYIMAGENET\": 4,\n}\n\n# Samplers evaluated in Phase 2\nSAMPLERS_TO_COMPARE: List[str] = [\n    \"stochastic\",\n    \"sobol\",\n    \"collatz_v1\",\n    \"collatz_v2\",\n    \"collatz_v3\",\n]\n\n_BASE: Dict[str, Any] = {\n    \"optimizer\":       \"SGD\",\n    \"momentum\":        0.9,\n    \"weight_decay\":    1e-4,\n    \"val_fraction\":    0.1,\n    \"device\":          \"auto\",\n    \"num_workers\":     2,\n    \"save_checkpoints\": True,\n    \"verbose\":         True,\n    \"samplers\":        SAMPLERS_TO_COMPARE,\n    \"dataset_model_map\": DATASET_MODEL_MAP,\n}\n\n_MODES: Dict[str, Dict[str, Any]] = {\n    \"DEBUG\": {\n        \"execution_mode\": \"DEBUG\",\n        \"datasets\":       [\"MNIST\"],\n        \"epochs\":         3,\n        \"seeds\":          [42, 43, 44],\n        \"batch_size\":     256,\n        \"lr\":             0.01,\n        \"lr_schedule\":    \"constant\",\n        \"output_dir\":     \"./dest_scaling_debug\",\n    },\n    \"VALIDATION\": {\n        \"execution_mode\": \"VALIDATION\",\n        \"datasets\":       [\"FASHIONMNIST\"],\n        \"epochs\":         10,\n        \"seeds\":          list(range(42, 52)),\n        \"batch_size\":     256,\n        \"lr\":             0.01,\n        \"lr_schedule\":    \"cosine\",\n        \"output_dir\":     \"./dest_scaling_validation\",\n    },\n    \"PAPER\": {\n        \"execution_mode\": \"PAPER\",\n        \"datasets\":       [\"CIFAR10\"],\n        \"epochs\":         15,\n        \"seeds\":          list(range(42, 62)),\n        \"batch_size\":     128,\n        \"lr\":             0.01,\n        \"lr_schedule\":    \"cosine\",\n        \"output_dir\":     \"./dest_scaling_paper\",\n    },\n    \"FULL\": {\n        \"execution_mode\": \"FULL\",\n        \"datasets\":       [\"MNIST\", \"FASHIONMNIST\", \"CIFAR10\", \"CIFAR100\", \"TINYIMAGENET\"],\n        \"epochs\":         20,\n        \"seeds\":          list(range(42, 62)),\n        \"batch_size\":     128,\n        \"lr\":             0.01,\n        \"lr_schedule\":    \"cosine\",\n        \"output_dir\":     \"./dest_scaling_full\",\n    },\n}\n\n\ndef get_config(\n    mode: str = \"DEBUG\",\n    dataset_override: Optional[str] = None,\n    model_override: Optional[str] = None,\n) -> Dict[str, Any]:\n    \"\"\"Return a fully-populated configuration dict for the given execution mode.\"\"\"\n    key = mode.upper()\n    if key not in _MODES:\n        raise ValueError(f\"Unknown mode '{mode}'. Choose from: {list(_MODES.keys())}\")\n    cfg = {**_BASE, **_MODES[key]}\n    if dataset_override:\n        cfg[\"datasets\"] = [dataset_override.upper()]\n    if model_override:\n        cfg[\"model_override\"] = model_override\n    return cfg\n",
    "datasets.py": "\"\"\"\ndatasets.py \u2014 Dataset loading with proper augmentation and TinyImageNet support.\n\nAugmentation policy (train only; test uses only normalization):\n  MNIST / FashionMNIST : none (28x28, grayscale, easy enough)\n  CIFAR-10  : RandomCrop(32, pad=4) + RandomHorizontalFlip\n  CIFAR-100 : RandomCrop(32, pad=4) + RandomHorizontalFlip\n  TinyImageNet : RandomCrop(64, pad=8) + RandomHorizontalFlip\n\nAll augmentations are applied identically to every sampler \u2014 the only\nvariable between experiments is the ORDER in which samples are presented.\n\"\"\"\n\nimport os\nimport shutil\nimport urllib.request\nimport zipfile\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader, Subset\nfrom torchvision import datasets, transforms\nfrom sklearn.model_selection import train_test_split\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 normalization constants \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n_STATS = {\n    \"MNIST\":        {\"mean\": (0.1307,),                  \"std\": (0.3081,)},\n    \"FASHIONMNIST\": {\"mean\": (0.2860,),                  \"std\": (0.3530,)},\n    \"CIFAR10\":      {\"mean\": (0.4914, 0.4822, 0.4465),   \"std\": (0.2023, 0.1994, 0.2010)},\n    \"CIFAR100\":     {\"mean\": (0.5071, 0.4867, 0.4408),   \"std\": (0.2675, 0.2565, 0.2761)},\n    \"TINYIMAGENET\": {\"mean\": (0.4802, 0.4481, 0.3975),   \"std\": (0.2302, 0.2265, 0.2262)},\n}\n\n\ndef _build_transforms(dataset_name: str, input_size: int):\n    \"\"\"Return (train_transform, test_transform) for the given dataset.\"\"\"\n    name = dataset_name.upper()\n    mean = _STATS[name][\"mean\"]\n    std  = _STATS[name][\"std\"]\n\n    normalize = transforms.Normalize(mean, std)\n\n    if name in (\"MNIST\", \"FASHIONMNIST\"):\n        base = [transforms.ToTensor(), normalize]\n        return transforms.Compose(base), transforms.Compose(base)\n\n    pad = input_size // 8  # 4 for 32-px, 8 for 64-px\n    train_tf = transforms.Compose([\n        transforms.RandomCrop(input_size, padding=pad),\n        transforms.RandomHorizontalFlip(),\n        transforms.ToTensor(),\n        normalize,\n    ])\n    test_tf = transforms.Compose([transforms.ToTensor(), normalize])\n    return train_tf, test_tf\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 TinyImageNet helpers \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n_TINYIMAGENET_URL = \"http://cs231n.stanford.edu/tiny-imagenet-200.zip\"\n\n\ndef _organize_tinyimagenet_val(data_dir: str) -> None:\n    \"\"\"Move TinyImageNet val images into per-class subdirectories.\"\"\"\n    val_dir = os.path.join(data_dir, \"val\")\n    ann_file = os.path.join(val_dir, \"val_annotations.txt\")\n    if not os.path.exists(ann_file):\n        return\n    # Check if already organized (skip if class dirs exist)\n    if any(\n        os.path.isdir(os.path.join(val_dir, d))\n        for d in os.listdir(val_dir)\n        if d.startswith(\"n\")\n    ):\n        return\n    print(\"Organizing TinyImageNet val set...\")\n    with open(ann_file) as f:\n        for line in f:\n            parts = line.strip().split(\"\\t\")\n            img_name, cls = parts[0], parts[1]\n            cls_dir = os.path.join(val_dir, cls, \"images\")\n            os.makedirs(cls_dir, exist_ok=True)\n            src = os.path.join(val_dir, \"images\", img_name)\n            dst = os.path.join(cls_dir, img_name)\n            if os.path.exists(src) and not os.path.exists(dst):\n                shutil.copy2(src, dst)\n\n\ndef download_tinyimagenet(root: str = \"./data\") -> str | None:\n    \"\"\"Download and extract TinyImageNet. Returns path or None on failure.\"\"\"\n    extract_path = os.path.join(root, \"tiny-imagenet-200\")\n    if os.path.exists(extract_path):\n        _organize_tinyimagenet_val(extract_path)\n        return extract_path\n    os.makedirs(root, exist_ok=True)\n    zip_path = os.path.join(root, \"tiny-imagenet-200.zip\")\n    try:\n        print(f\"Downloading TinyImageNet (~237 MB)\u2026\")\n        urllib.request.urlretrieve(_TINYIMAGENET_URL, zip_path)\n        print(\"Extracting\u2026\")\n        with zipfile.ZipFile(zip_path, \"r\") as zf:\n            zf.extractall(root)\n        os.remove(zip_path)\n        _organize_tinyimagenet_val(extract_path)\n        return extract_path\n    except Exception as exc:\n        print(f\"\u26a0\ufe0f  TinyImageNet download failed ({exc}). Skipping dataset.\")\n        if os.path.exists(zip_path):\n            os.remove(zip_path)\n        return None\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 main loader class \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nclass DatasetLoader:\n    \"\"\"\n    Unified dataset loader.  Returns stratified train / val / test splits with\n    correct per-dataset augmentation and normalization.\n\n    Parameters\n    ----------\n    dataset_name : str\n        One of MNIST, FASHIONMNIST, CIFAR10, CIFAR100, TINYIMAGENET.\n    val_fraction : float\n        Fraction of training data reserved for validation.\n    split_seed : int\n        Seed for the deterministic stratified split (same across all samplers).\n    data_root : str\n        Directory where datasets are downloaded / cached.\n    \"\"\"\n\n    def __init__(\n        self,\n        dataset_name: str = \"MNIST\",\n        val_fraction: float = 0.1,\n        split_seed: int = 0,\n        data_root: str = \"./data\",\n    ):\n        self.dataset_name = dataset_name.upper()\n        self.val_fraction = val_fraction\n        self.split_seed = split_seed\n        self.data_root = data_root\n\n    # ------------------------------------------------------------------\n    def get_datasets(self):\n        \"\"\"\n        Returns\n        -------\n        train_dataset, val_dataset, test_dataset, n_classes, input_shape, available\n        available : bool  \u2014 False only for TinyImageNet when download fails.\n        \"\"\"\n        name = self.dataset_name\n\n        if name == \"TINYIMAGENET\":\n            return self._load_tinyimagenet()\n\n        # \u2500\u2500 standard torchvision datasets \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        input_size = 28 if name in (\"MNIST\", \"FASHIONMNIST\") else 32\n        train_tf, test_tf = _build_transforms(name, input_size)\n\n        kw = {\"root\": self.data_root, \"download\": True}\n        if name == \"MNIST\":\n            train_full   = datasets.MNIST(train=True,  transform=train_tf, **kw)\n            test_dataset = datasets.MNIST(train=False, transform=test_tf,  **kw)\n            n_classes, input_shape = 10, (1, 28, 28)\n\n        elif name == \"FASHIONMNIST\":\n            train_full   = datasets.FashionMNIST(train=True,  transform=train_tf, **kw)\n            test_dataset = datasets.FashionMNIST(train=False, transform=test_tf,  **kw)\n            n_classes, input_shape = 10, (1, 28, 28)\n\n        elif name == \"CIFAR10\":\n            train_full   = datasets.CIFAR10(train=True,  transform=train_tf, **kw)\n            test_dataset = datasets.CIFAR10(train=False, transform=test_tf,  **kw)\n            n_classes, input_shape = 10, (3, 32, 32)\n\n        elif name == \"CIFAR100\":\n            train_full   = datasets.CIFAR100(train=True,  transform=train_tf, **kw)\n            test_dataset = datasets.CIFAR100(train=False, transform=test_tf,  **kw)\n            n_classes, input_shape = 100, (3, 32, 32)\n\n        else:\n            raise ValueError(f\"Unknown dataset: {self.dataset_name}\")\n\n        train_ds, val_ds = self._stratified_split(train_full)\n        return train_ds, val_ds, test_dataset, n_classes, input_shape, True\n\n    # ------------------------------------------------------------------\n    def _load_tinyimagenet(self):\n        data_dir = download_tinyimagenet(self.data_root)\n        if data_dir is None:\n            return None, None, None, None, None, False  # auto-skip\n\n        train_tf, test_tf = _build_transforms(\"TINYIMAGENET\", 64)\n        from torchvision.datasets import ImageFolder\n\n        train_root = os.path.join(data_dir, \"train\")\n        val_root   = os.path.join(data_dir, \"val\")\n\n        train_full   = ImageFolder(train_root, transform=train_tf)\n        test_dataset = ImageFolder(val_root,   transform=test_tf)\n\n        n_classes   = len(train_full.classes)\n        input_shape = (3, 64, 64)\n\n        train_ds, val_ds = self._stratified_split(train_full)\n        return train_ds, val_ds, test_dataset, n_classes, input_shape, True\n\n    # ------------------------------------------------------------------\n    def _stratified_split(self, train_full):\n        targets = getattr(train_full, \"targets\", None)\n        if targets is None:\n            # ImageFolder stores .targets as a list\n            targets = [s[1] for s in train_full.samples]\n        if torch.is_tensor(targets):\n            targets = targets.numpy()\n        targets = np.array(targets)\n\n        train_idx, val_idx = train_test_split(\n            np.arange(len(train_full)),\n            test_size=self.val_fraction,\n            random_state=self.split_seed,\n            stratify=targets,\n        )\n        return Subset(train_full, train_idx), Subset(train_full, val_idx)\n",
    "manifest.py": "import os\nimport json\nimport time\nimport datetime\nimport torch\n\nclass ExperimentManifest:\n    def __init__(self, output_dir: str):\n        self.output_dir = output_dir\n        os.makedirs(output_dir, exist_ok=True)\n        self.manifest_path = os.path.join(output_dir, \"experiment_manifest.json\")\n        self.manifest = self._load()\n\n    def _load(self):\n        if os.path.exists(self.manifest_path):\n            try:\n                with open(self.manifest_path, 'r') as f:\n                    return json.load(f)\n            except Exception:\n                return {\"experiments\": {}, \"system_info\": self._get_sys_info()}\n        return {\"experiments\": {}, \"system_info\": self._get_sys_info()}\n\n    def _get_sys_info(self):\n        return {\n            \"timestamp\": datetime.datetime.now().isoformat(),\n            \"cuda_available\": torch.cuda.is_available(),\n            \"gpu_name\": torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"CPU\",\n            \"pytorch_version\": torch.__version__\n        }\n\n    def save(self):\n        with open(self.manifest_path, 'w') as f:\n            json.dump(self.manifest, f, indent=4)\n\n    def is_experiment_complete(self, exp_id: str, n_seeds: int) -> bool:\n        if exp_id in self.manifest[\"experiments\"]:\n            exp_data = self.manifest[\"experiments\"][exp_id]\n            return len(exp_data.get(\"completed_seeds\", [])) >= n_seeds\n        return False\n\n    def is_seed_complete(self, exp_id: str, seed: int) -> bool:\n        if exp_id in self.manifest[\"experiments\"]:\n            return seed in self.manifest[\"experiments\"][exp_id].get(\"completed_seeds\", [])\n        return False\n\n    def mark_seed_complete(self, exp_id: str, seed: int):\n        if exp_id not in self.manifest[\"experiments\"]:\n            self.manifest[\"experiments\"][exp_id] = {\"completed_seeds\": [], \"status\": \"IN_PROGRESS\"}\n        if seed not in self.manifest[\"experiments\"][exp_id][\"completed_seeds\"]:\n            self.manifest[\"experiments\"][exp_id][\"completed_seeds\"].append(seed)\n        self.save()\n\n    def mark_experiment_complete(self, exp_id: str):\n        if exp_id in self.manifest[\"experiments\"]:\n            self.manifest[\"experiments\"][exp_id][\"status\"] = \"COMPLETED\"\n        else:\n            self.manifest[\"experiments\"][exp_id] = {\"completed_seeds\": [], \"status\": \"COMPLETED\"}\n        self.save()\n",
    "metrics.py": "\"\"\"\nmetrics.py \u2014 Model evaluation utilities.\n\nFixes vs v1:\n  - compute_ece now correctly operates on the full 2-D probability matrix\n    (predictions vs. class indices).  ECE is now in [0, 1].\n  - evaluate_model passes the full prob matrix to compute_ece.\n\"\"\"\n\nimport numpy as np\nimport torch\nfrom sklearn.metrics import f1_score, precision_score, recall_score\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\ndef compute_ece(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 15) -> float:\n    \"\"\"\n    Expected Calibration Error (ECE).\n\n    Parameters\n    ----------\n    y_true : (N,) array of true class indices.\n    y_prob : (N, C) array of class probabilities  *or*\n             (N,)   array of max confidences (legacy; accuracy unavailable).\n\n    Returns\n    -------\n    float in [0, 1].\n    \"\"\"\n    if y_prob.ndim == 2:\n        confidence  = np.max(y_prob, axis=1)            # (N,)\n        predictions = np.argmax(y_prob, axis=1)          # (N,)\n        correct     = (predictions == y_true).astype(float)\n    else:\n        # Legacy path: only max-confidence scalar per sample; accuracy unknown.\n        # Return a simple mean-absolute-deviation as a proxy.\n        confidence = y_prob\n        correct    = np.zeros_like(confidence)           # can't know accuracy\n\n    n   = len(y_true)\n    ece = 0.0\n    bins = np.linspace(0.0, 1.0, n_bins + 1)\n\n    for lo, hi in zip(bins[:-1], bins[1:]):\n        mask = (confidence > lo) & (confidence <= hi)\n        if mask.sum() == 0:\n            continue\n        avg_conf = confidence[mask].mean()\n        avg_acc  = correct[mask].mean()\n        ece += mask.sum() * abs(avg_conf - avg_acc)\n\n    return float(ece / max(1, n))\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\ndef evaluate_model(model, device, dataloader, criterion):\n    \"\"\"\n    Run one full pass over `dataloader` in eval mode.\n\n    Returns\n    -------\n    avg_loss, accuracy_pct, f1_macro, precision_macro, recall_macro, ece\n    \"\"\"\n    model.eval()\n    total_loss  = 0.0\n    all_preds   = []\n    all_targets = []\n    all_probs   = []          # full softmax distributions\n\n    with torch.no_grad():\n        for data, target in dataloader:\n            data, target = data.to(device), target.to(device)\n            output = model(data)\n            loss   = criterion(output, target)\n            total_loss += loss.item() * data.size(0)\n\n            probs = torch.softmax(output, dim=1)\n            preds = output.argmax(dim=1)\n\n            all_preds.extend(preds.cpu().numpy())\n            all_targets.extend(target.cpu().numpy())\n            all_probs.extend(probs.cpu().numpy())\n\n    n           = len(all_targets)\n    avg_loss    = total_loss / max(1, n)\n    all_preds   = np.array(all_preds)\n    all_targets = np.array(all_targets)\n    all_probs   = np.array(all_probs)          # (N, C)\n\n    acc       = float((all_preds == all_targets).mean() * 100.0)\n    f1        = float(f1_score(all_targets, all_preds, average=\"macro\", zero_division=0))\n    precision = float(precision_score(all_targets, all_preds, average=\"macro\", zero_division=0))\n    recall    = float(recall_score(all_targets, all_preds, average=\"macro\", zero_division=0))\n    ece       = compute_ece(all_targets, all_probs)       # pass full (N,C) matrix\n\n    return avg_loss, acc, f1, precision, recall, ece\n",
    "models.py": "\"\"\"\nmodels.py \u2014 Model factory for all DEST experiments.\n\nModels:\n  SmallCNN  \u2014 2-conv + FC, for MNIST / FashionMNIST (28x28 grayscale)\n  ResNet9   \u2014 lightweight 9-layer residual, for CIFAR-10 (32x32)\n  ResNet18  \u2014 standard torchvision ResNet-18, adapted for 32x32 (CIFAR-100)\n              or 64x64 (TinyImageNet)\n\nDeterministicDropout: shared deterministic / stochastic dropout implementation.\n\"\"\"\n\nimport torch\nimport torch.nn as nn\nfrom torchvision.models import resnet18\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 shared dropout module \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nclass DeterministicDropout(nn.Module):\n    \"\"\"\n    Drop-in replacement for nn.Dropout that supports two modes:\n\n    'stochastic' : standard random dropout (identical to nn.Dropout).\n    'deterministic' : rotating binary mask \u2014 keeps exactly (1-p) fraction\n                      of features, cycling the mask position each step.\n                      This gives deterministic diversity without random noise.\n    \"\"\"\n\n    def __init__(self, p: float = 0.5, mode: str = \"stochastic\"):\n        super().__init__()\n        self.p    = p\n        self.mode = mode\n        self.register_buffer(\"step_counter\", torch.zeros(1, dtype=torch.long))\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        if not self.training or self.p == 0:\n            return x\n\n        if self.mode == \"stochastic\":\n            mask = (torch.rand_like(x) >= self.p).float()\n            return x * mask / (1.0 - self.p)\n\n        elif self.mode == \"deterministic\":\n            _, num_features = x.shape[0], x.shape[1]\n            keep_n  = max(1, int(round((1.0 - self.p) * num_features)))\n            base    = torch.zeros(num_features, device=x.device)\n            base[:keep_n] = 1.0\n            step    = int(self.step_counter.item())\n            batch_size = x.shape[0]\n            masks   = torch.stack(\n                [torch.roll(base, (step + i) % num_features, 0) for i in range(batch_size)]\n            )\n            self.step_counter += 1\n            scale = num_features / keep_n\n            return x * masks * scale\n\n        else:\n            raise ValueError(f\"Unknown dropout mode: {self.mode}\")\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 SmallCNN (MNIST / FashionMNIST) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nclass SmallCNN(nn.Module):\n    def __init__(\n        self,\n        in_channels: int = 1,\n        num_classes: int = 10,\n        dropout_mode: str = \"stochastic\",\n        dropout_prob: float = 0.5,\n    ):\n        super().__init__()\n        self.features = nn.Sequential(\n            nn.Conv2d(in_channels, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),\n            nn.Conv2d(8, 16, 3, padding=1),          nn.ReLU(), nn.MaxPool2d(2),\n        )\n        self.classifier = nn.Sequential(\n            nn.Flatten(),\n            nn.Linear(16 * 7 * 7, 128),\n            nn.ReLU(),\n            DeterministicDropout(p=dropout_prob, mode=dropout_mode),\n            nn.Linear(128, num_classes),\n        )\n\n    def forward(self, x):\n        return self.classifier(self.features(x))\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 ResNet9 (CIFAR-10) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\ndef _conv_block(in_ch, out_ch, pool=False):\n    layers = [\n        nn.Conv2d(in_ch, out_ch, 3, padding=1),\n        nn.BatchNorm2d(out_ch),\n        nn.ReLU(inplace=True),\n    ]\n    if pool:\n        layers.append(nn.MaxPool2d(2))\n    return nn.Sequential(*layers)\n\n\nclass ResNet9(nn.Module):\n    def __init__(\n        self,\n        in_channels: int = 3,\n        num_classes: int = 10,\n        dropout_mode: str = \"stochastic\",\n        dropout_prob: float = 0.2,\n    ):\n        super().__init__()\n        self.prep   = _conv_block(in_channels, 64)\n        self.layer1 = nn.Sequential(_conv_block(64, 128, pool=True),\n                                    nn.Sequential(_conv_block(128, 128), _conv_block(128, 128)))\n        self.layer2 = _conv_block(128, 256, pool=True)\n        self.layer3 = nn.Sequential(_conv_block(256, 512, pool=True),\n                                    nn.Sequential(_conv_block(512, 512), _conv_block(512, 512)))\n        self.head   = nn.Sequential(\n            nn.MaxPool2d(4),\n            nn.Flatten(),\n            DeterministicDropout(p=dropout_prob, mode=dropout_mode),\n            nn.Linear(512, num_classes),\n        )\n\n    def forward(self, x):\n        return self._fwd(x)\n\n    def _fwd(self, x):\n        x  = self.prep(x)\n        x1 = self.layer1[0](x)\n        x  = self.layer1[1](x1) + x1\n        x  = self.layer2(x)\n        x3 = self.layer3[0](x)\n        x  = self.layer3[1](x3) + x3\n        return self.head(x)\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 ResNet18 (CIFAR-100 / TinyImageNet) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\ndef _build_resnet18(in_channels: int, num_classes: int, input_size: int) -> nn.Module:\n    \"\"\"\n    Torchvision ResNet-18 adapted for the given spatial input size.\n    - 32x32 (CIFAR-100): replace 7x7/stride-2 conv with 3x3/stride-1, remove maxpool.\n    - 64x64 (TinyImageNet): keep standard first conv, remove maxpool.\n    - Others: standard.\n    \"\"\"\n    model = resnet18(weights=None, num_classes=num_classes)\n    if in_channels != 3:\n        model.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)\n    if input_size <= 32:\n        model.conv1  = nn.Conv2d(in_channels, 64, kernel_size=3, stride=1, padding=1, bias=False)\n        model.maxpool = nn.Identity()\n    elif input_size <= 64:\n        model.maxpool = nn.Identity()\n    return model\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 SimpleMLP (fallback) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nclass SimpleMLP(nn.Module):\n    def __init__(self, in_features=784, num_classes=10, dropout_mode=\"stochastic\", dropout_prob=0.5):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Flatten(),\n            nn.Linear(in_features, 256), nn.ReLU(),\n            DeterministicDropout(p=dropout_prob, mode=dropout_mode),\n            nn.Linear(256, 128), nn.ReLU(),\n            DeterministicDropout(p=dropout_prob, mode=dropout_mode),\n            nn.Linear(128, num_classes),\n        )\n    def forward(self, x):\n        return self.net(x)\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 factory \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nclass ModelFactory:\n    @staticmethod\n    def get_model(\n        model_name: str,\n        input_shape: tuple,\n        num_classes: int,\n        dropout_mode: str = \"stochastic\",\n    ) -> nn.Module:\n        name       = model_name.lower()\n        in_ch      = input_shape[0]\n        input_size = input_shape[1]          # height (assume square)\n\n        if name in (\"smallcnn\", \"cnn\"):\n            return SmallCNN(in_channels=in_ch, num_classes=num_classes, dropout_mode=dropout_mode)\n\n        elif name == \"resnet9\":\n            return ResNet9(in_channels=in_ch, num_classes=num_classes, dropout_mode=dropout_mode)\n\n        elif name in (\"resnet18\",):\n            return _build_resnet18(in_ch, num_classes, input_size)\n\n        elif name in (\"simplemlp\", \"mlp\"):\n            in_features = int(in_ch * input_shape[1] * input_shape[2])\n            return SimpleMLP(in_features=in_features, num_classes=num_classes, dropout_mode=dropout_mode)\n\n        else:\n            raise ValueError(f\"Unknown model: {model_name}\")\n",
    "plotting.py": "\"\"\"\nplotting.py \u2014 Publication-quality figures for DEST experiments.\n\nNew in v2:\n  - Variance bands (mean \u00b1 std across seeds) on accuracy/loss curves.\n  - plot_dataset_scaling: the central Phase-2 figure.\n  - plot_final_comparison: grouped bar chart across methods per dataset.\n  - plot_convergence_speed: bar chart of convergence epoch.\n  - All figures saved as high-res PNG (300 dpi) + vector PDF.\n\"\"\"\n\nimport os\nimport warnings\n\nimport matplotlib\nmatplotlib.use(\"Agg\")\nimport matplotlib.pyplot as plt\nimport matplotlib.patches as mpatches\nimport numpy as np\nimport seaborn as sns\n\nfrom typing import Dict, List, Any, Optional\n\nwarnings.filterwarnings(\"ignore\", category=UserWarning)\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 style constants \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nPALETTE = {\n    \"stochastic\":  \"#607D8B\",\n    \"sobol\":       \"#2196F3\",\n    \"collatz_v1\":  \"#FF9800\",\n    \"collatz_v2\":  \"#9C27B0\",\n    \"collatz_v3\":  \"#F44336\",\n}\nLABELS = {\n    \"stochastic\":  \"Random Shuffle (baseline)\",\n    \"sobol\":       \"Sobol Scrambled\",\n    \"collatz_v1\":  \"Collatz V1\",\n    \"collatz_v2\":  \"Collatz V2\",\n    \"collatz_v3\":  \"Collatz V3 (DEST)\",\n}\nDATASET_LABELS = {\n    \"MNIST\":        \"MNIST\\n(easy)\",\n    \"FASHIONMNIST\": \"Fashion-MNIST\\n(moderate)\",\n    \"CIFAR10\":      \"CIFAR-10\\n(hard)\",\n    \"CIFAR100\":     \"CIFAR-100\\n(harder)\",\n    \"TINYIMAGENET\": \"TinyImageNet\\n(hardest)\",\n}\n\nplt.rcParams.update({\n    \"font.family\":   \"DejaVu Sans\",\n    \"axes.titlesize\": 13,\n    \"axes.labelsize\": 11,\n    \"legend.fontsize\": 9,\n    \"figure.dpi\":    100,\n})\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 helper \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\ndef _save(fig, paths):\n    \"\"\"Save figure to every path in `paths`.\"\"\"\n    for p in paths:\n        os.makedirs(os.path.dirname(p), exist_ok=True)\n        fig.savefig(p, dpi=300, bbox_inches=\"tight\")\n    plt.close(fig)\n\n\ndef _runs_to_matrix(runs, key=\"test_accs\"):\n    \"\"\"Stack per-epoch curves from a list of RunResult dicts -> (seeds, epochs) array.\"\"\"\n    arr = [getattr(r, key, None) or r.get(key, []) for r in runs]\n    # normalise length\n    L = min(len(a) for a in arr) if arr else 0\n    return np.array([a[:L] for a in arr]) if L else np.empty((0, 0))\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500 Plotter class \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nclass Plotter:\n    def __init__(self, output_dir: str):\n        self.output_dir = output_dir\n        self.plots_dir  = os.path.join(output_dir, \"plots\")\n        self.paper_dir  = os.path.join(output_dir, \"paper\", \"figures\")\n        os.makedirs(self.plots_dir, exist_ok=True)\n        os.makedirs(self.paper_dir, exist_ok=True)\n\n    # \u2500\u2500 variance-band accuracy curves \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    def plot_accuracy_curves(self, results_by_sampler: Dict, tag: str = \"\"):\n        fig, ax = plt.subplots(figsize=(9, 5))\n        for sampler, runs in results_by_sampler.items():\n            mat = _runs_to_matrix(runs, \"test_accs\")\n            if mat.size == 0:\n                continue\n            mu  = mat.mean(0)\n            sd  = mat.std(0)\n            ep  = np.arange(1, len(mu) + 1)\n            c   = PALETTE.get(sampler, \"#333333\")\n            ax.plot(ep, mu, color=c, label=LABELS.get(sampler, sampler), linewidth=2)\n            ax.fill_between(ep, mu - sd, mu + sd, alpha=0.15, color=c)\n\n        ax.set_xlabel(\"Epoch\"); ax.set_ylabel(\"Test Accuracy (%)\")\n        ax.set_title(f\"Test Accuracy per Epoch \u2014 {tag}\")\n        ax.legend(loc=\"lower right\"); ax.grid(alpha=0.3)\n        stem = f\"{tag}_accuracy_curves\" if tag else \"accuracy_curves\"\n        _save(fig, [f\"{self.plots_dir}/{stem}.png\", f\"{self.paper_dir}/{stem}.pdf\"])\n\n    # \u2500\u2500 variance-band loss curves \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    def plot_loss_curves(self, results_by_sampler: Dict, tag: str = \"\"):\n        fig, ax = plt.subplots(figsize=(9, 5))\n        for sampler, runs in results_by_sampler.items():\n            mat = _runs_to_matrix(runs, \"train_losses\")\n            if mat.size == 0:\n                continue\n            mu  = mat.mean(0); sd = mat.std(0)\n            ep  = np.arange(1, len(mu) + 1)\n            c   = PALETTE.get(sampler, \"#333333\")\n            ax.plot(ep, mu, color=c, label=LABELS.get(sampler, sampler), linewidth=2)\n            ax.fill_between(ep, mu - sd, mu + sd, alpha=0.15, color=c)\n\n        ax.set_xlabel(\"Epoch\"); ax.set_ylabel(\"Train Loss\")\n        ax.set_title(f\"Train Loss per Epoch \u2014 {tag}\")\n        ax.legend(); ax.grid(alpha=0.3)\n        stem = f\"{tag}_loss_curves\" if tag else \"loss_curves\"\n        _save(fig, [f\"{self.plots_dir}/{stem}.png\", f\"{self.paper_dir}/{stem}.pdf\"])\n\n    # \u2500\u2500 box-plots of final accuracy \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    def plot_boxplots(self, results_by_sampler: Dict, tag: str = \"\"):\n        samplers = list(results_by_sampler.keys())\n        data = [[r.final_test_acc if hasattr(r, \"final_test_acc\") else r.get(\"final_test_acc\", 0)\n                 for r in results_by_sampler[s]] for s in samplers]\n\n        fig, ax = plt.subplots(figsize=(max(6, len(samplers) * 1.8), 5))\n        bp = ax.boxplot(data, patch_artist=True, medianprops={\"color\": \"black\", \"linewidth\": 2})\n        for patch, s in zip(bp[\"boxes\"], samplers):\n            patch.set_facecolor(PALETTE.get(s, \"#90A4AE\"))\n            patch.set_alpha(0.7)\n        ax.set_xticks(range(1, len(samplers) + 1))\n        ax.set_xticklabels([LABELS.get(s, s) for s in samplers], rotation=20, ha=\"right\")\n        ax.set_ylabel(\"Test Accuracy (%)\")\n        ax.set_title(f\"Final Test Accuracy Distribution \u2014 {tag}\")\n        ax.grid(alpha=0.3, axis=\"y\")\n        stem = f\"{tag}_boxplots\" if tag else \"boxplots\"\n        _save(fig, [f\"{self.plots_dir}/{stem}.png\", f\"{self.paper_dir}/{stem}.pdf\"])\n\n    # \u2500\u2500 randomness sweep (Exp J) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    def plot_randomness_sweep(self, sweep_results: Dict):\n        alphas = sorted(sweep_results.keys())\n        means  = [np.mean([r.final_test_acc for r in sweep_results[a]]) for a in alphas]\n        stds   = [np.std( [r.final_test_acc for r in sweep_results[a]]) for a in alphas]\n\n        fig, ax = plt.subplots(figsize=(8, 5))\n        ax.errorbar(alphas, means, yerr=stds, marker=\"o\", linewidth=2, capsize=5, color=\"#E91E63\")\n        ax.set_xlabel(\"\u03b1 (0 = pure Collatz, 1 = pure random)\"); ax.set_ylabel(\"Test Accuracy (%)\")\n        ax.set_title(\"Randomness Sweep \u2014 Effect of \u03b1 on Accuracy\")\n        ax.grid(alpha=0.3)\n        _save(fig, [f\"{self.plots_dir}/exp_J_randomness_sweep.png\",\n                    f\"{self.paper_dir}/exp_J_randomness_sweep.pdf\"])\n\n    # \u2500\u2500 \u2605 DATASET SCALING PLOT (Phase 2 key figure) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    def plot_dataset_scaling(\n        self,\n        all_results: Dict,           # keyed by (dataset, sampler_name)\n        dataset_order: List[str],\n        output_tag: str = \"DEST_scaling_validation\",\n    ):\n        \"\"\"\n        X-axis : Dataset difficulty (MNIST \u2192 TinyImageNet)\n        Y-axis : Test accuracy improvement over stochastic baseline (\u0394 pp)\n\n        One line per non-baseline sampler, with \u00b11\u03c3 band across seeds.\n        The horizontal reference line at y=0 is the stochastic baseline.\n        \"\"\"\n        comparators = [\"sobol\", \"collatz_v1\", \"collatz_v2\", \"collatz_v3\"]\n        available_datasets = [\n            d for d in dataset_order if any((d, s) in all_results for s in comparators)\n        ]\n        if not available_datasets:\n            print(\"\u26a0\ufe0f  No data available for scaling plot.\"); return\n\n        fig, ax = plt.subplots(figsize=(11, 6))\n\n        for sampler in comparators:\n            x_vals, y_means, y_stds = [], [], []\n            for i, ds in enumerate(available_datasets):\n                baseline_runs  = all_results.get((ds, \"stochastic\"), [])\n                treatment_runs = all_results.get((ds, sampler), [])\n                if not baseline_runs or not treatment_runs:\n                    continue\n                base_acc = np.mean([r.final_test_acc for r in baseline_runs])\n                trt_accs = np.array([r.final_test_acc for r in treatment_runs])\n                x_vals.append(i)\n                y_means.append(float(np.mean(trt_accs) - base_acc))\n                y_stds.append(float(np.std(trt_accs)))\n\n            if not x_vals:\n                continue\n            c  = PALETTE[sampler]\n            lbl = LABELS[sampler]\n            ax.plot(x_vals, y_means, \"o-\", color=c, label=lbl, linewidth=2.5, markersize=9)\n            ax.fill_between(\n                x_vals,\n                np.array(y_means) - np.array(y_stds),\n                np.array(y_means) + np.array(y_stds),\n                alpha=0.15, color=c,\n            )\n\n        ax.axhline(0, color=\"#455A64\", linestyle=\"--\", linewidth=1.5,\n                   label=\"Stochastic baseline (\u0394 = 0)\")\n        ax.set_xticks(range(len(available_datasets)))\n        ax.set_xticklabels(\n            [DATASET_LABELS.get(d, d) for d in available_datasets], fontsize=11\n        )\n        ax.set_xlabel(\"Dataset Difficulty \u2192\", fontsize=13)\n        ax.set_ylabel(\"Test Accuracy Improvement over Random (pp)\", fontsize=13)\n        ax.set_title(\n            \"DEST Phase 2 \u2014 Does improvement scale with dataset difficulty?\",\n            fontsize=14, fontweight=\"bold\",\n        )\n        ax.legend(loc=\"upper left\", framealpha=0.9)\n        ax.grid(alpha=0.3)\n        ax.tick_params(axis=\"both\", labelsize=11)\n\n        _save(fig, [f\"{self.plots_dir}/{output_tag}.png\",\n                    f\"{self.paper_dir}/{output_tag}.pdf\"])\n        print(f\"\u2705 Scaling plot saved \u2192 {self.plots_dir}/{output_tag}.png\")\n\n    # \u2500\u2500 final accuracy comparison (grouped bar) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    def plot_final_comparison(self, all_results: Dict, dataset_order: List[str]):\n        samplers = [\"stochastic\", \"sobol\", \"collatz_v1\", \"collatz_v2\", \"collatz_v3\"]\n        avail_ds = [d for d in dataset_order if any((d, s) in all_results for s in samplers)]\n        if not avail_ds:\n            return\n\n        fig, axes = plt.subplots(1, len(avail_ds), figsize=(5 * len(avail_ds), 6), sharey=False)\n        if len(avail_ds) == 1:\n            axes = [axes]\n\n        for ax, ds in zip(axes, avail_ds):\n            labels, means, errors, colors = [], [], [], []\n            for s in samplers:\n                runs = all_results.get((ds, s), [])\n                if not runs: continue\n                accs = [r.final_test_acc for r in runs]\n                labels.append(s.replace(\"_\", \"\\n\")); means.append(np.mean(accs))\n                errors.append(np.std(accs)); colors.append(PALETTE[s])\n            if not means: continue\n            x = np.arange(len(labels))\n            bars = ax.bar(x, means, yerr=errors, capsize=5, color=colors, alpha=0.8)\n            ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8)\n            ax.set_title(ds); ax.set_ylabel(\"Test Acc (%)\"); ax.grid(alpha=0.3, axis=\"y\")\n            ymin = max(0, min(means) - 2); ymax = max(means) + 1\n            ax.set_ylim(ymin, ymax)\n\n        fig.suptitle(\"Final Test Accuracy by Method and Dataset\", fontsize=14, fontweight=\"bold\")\n        plt.tight_layout()\n        _save(fig, [f\"{self.plots_dir}/final_comparison.png\",\n                    f\"{self.paper_dir}/final_comparison.pdf\"])\n\n    # \u2500\u2500 convergence speed \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    def plot_convergence_speed(self, all_results: Dict, dataset_order: List[str]):\n        samplers = [\"stochastic\", \"sobol\", \"collatz_v1\", \"collatz_v2\", \"collatz_v3\"]\n        avail_ds = [d for d in dataset_order if any((d, s) in all_results for s in samplers)]\n        if not avail_ds:\n            return\n\n        fig, ax = plt.subplots(figsize=(max(8, len(avail_ds) * 3), 5))\n        x = np.arange(len(avail_ds))\n        w = 0.15\n        for i, s in enumerate(samplers):\n            speeds = []\n            for ds in avail_ds:\n                runs = all_results.get((ds, s), [])\n                conv = [r.convergence_epoch_90 for r in runs if r.convergence_epoch_90]\n                speeds.append(np.mean(conv) if conv else None)\n            y = [v if v is not None else 0 for v in speeds]\n            ax.bar(x + i * w, y, w, label=LABELS[s], color=PALETTE[s], alpha=0.8)\n\n        ax.set_xticks(x + w * 2); ax.set_xticklabels(avail_ds)\n        ax.set_ylabel(\"Epoch to reach 90% test accuracy\")\n        ax.set_title(\"Convergence Speed by Method and Dataset\")\n        ax.legend(); ax.grid(alpha=0.3, axis=\"y\")\n        _save(fig, [f\"{self.plots_dir}/convergence_speed.png\",\n                    f\"{self.paper_dir}/convergence_speed.pdf\"])\n\n    # \u2500\u2500 seed variance heatmap \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    def plot_seed_variance(self, all_results: Dict, dataset_order: List[str]):\n        samplers = [\"stochastic\", \"sobol\", \"collatz_v1\", \"collatz_v2\", \"collatz_v3\"]\n        avail_ds = [d for d in dataset_order if any((d, s) in all_results for s in samplers)]\n        if not avail_ds:\n            return\n\n        matrix = np.zeros((len(samplers), len(avail_ds)))\n        for i, s in enumerate(samplers):\n            for j, ds in enumerate(avail_ds):\n                runs = all_results.get((ds, s), [])\n                accs = [r.final_test_acc for r in runs]\n                matrix[i, j] = np.std(accs) if len(accs) > 1 else 0.0\n\n        fig, ax = plt.subplots(figsize=(max(6, len(avail_ds) * 2), 4))\n        im = ax.imshow(matrix, cmap=\"YlOrRd\", aspect=\"auto\")\n        plt.colorbar(im, ax=ax, label=\"Std of test accuracy (pp)\")\n        ax.set_xticks(range(len(avail_ds))); ax.set_xticklabels(avail_ds)\n        ax.set_yticks(range(len(samplers))); ax.set_yticklabels([LABELS[s] for s in samplers])\n        ax.set_title(\"Cross-seed Variance (lower = more stable)\")\n        for i in range(len(samplers)):\n            for j in range(len(avail_ds)):\n                ax.text(j, i, f\"{matrix[i, j]:.3f}\", ha=\"center\", va=\"center\", fontsize=8)\n        _save(fig, [f\"{self.plots_dir}/seed_variance_heatmap.png\",\n                    f\"{self.paper_dir}/seed_variance_heatmap.pdf\"])\n",
    "report.py": "\"\"\"\nreport.py \u2014 Automated research report generator for Phase 2 scaling validation.\n\nAnswers the 6 core empirical questions:\n  Q1: Does DEST outperform Random?\n  Q2: Does DEST outperform Sobol?\n  Q3: Does improvement increase with dataset complexity?\n  Q4: Does DEST reduce variance?\n  Q5: Does DEST converge faster?\n  Q6: Is computational overhead acceptable?\n\"\"\"\n\nimport os\nimport json\nimport numpy as np\nfrom typing import Dict, List, Any\n\nfrom .statistics import Statistics\n\n\ndef _acc(runs):\n    return [r.final_test_acc for r in runs] if runs else []\n\ndef _sig(p):\n    return \"\u2705 YES (p={:.4f})\".format(p) if p < 0.05 else \"\u274c NO (p={:.4f})\".format(p)\n\n\nclass ReportGenerator:\n    def __init__(self, output_dir: str):\n        self.output_dir = output_dir\n        self.reports_dir = os.path.join(output_dir, \"reports\")\n        self.tables_dir  = os.path.join(output_dir, \"paper\", \"tables\")\n        os.makedirs(self.reports_dir, exist_ok=True)\n        os.makedirs(self.tables_dir,  exist_ok=True)\n\n    # \u2500\u2500 6-question automated analysis \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    def answer_six_questions(\n        self,\n        all_results: Dict,\n        dataset_order: List[str],\n        config: Dict,\n    ) -> Dict[str, str]:\n        \"\"\"\n        Parameters\n        ----------\n        all_results : dict keyed by (dataset, sampler_name) -> List[RunResult]\n        dataset_order : list of dataset names in difficulty order.\n\n        Returns\n        -------\n        dict {question_label: answer_text}\n        \"\"\"\n        dest = \"collatz_v3\"\n        baseline = \"stochastic\"\n        sobol    = \"sobol\"\n        avail = [d for d in dataset_order if (d, baseline) in all_results]\n\n        answers = {}\n\n        # \u2500\u2500 Q1 \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        q1_lines = [\"**Q1 \u2014 Does DEST outperform Random Shuffle?**\\n\"]\n        for ds in avail:\n            b_acc = _acc(all_results.get((ds, baseline), []))\n            d_acc = _acc(all_results.get((ds, dest), []))\n            if not b_acc or not d_acc: continue\n            cmp = Statistics.compare_groups(b_acc, d_acc)\n            delta = cmp[\"delta\"]\n            q1_lines.append(\n                f\"  {ds}: DEST={np.mean(d_acc):.3f}% vs Random={np.mean(b_acc):.3f}% \"\n                f\"(\u0394={delta:+.3f}pp, {_sig(cmp['p_val_ttest'])}, \"\n                f\"Cohen's d={cmp['cohens_d']:.2f})\"\n            )\n        answers[\"Q1\"] = \"\\n\".join(q1_lines)\n\n        # \u2500\u2500 Q2 \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        q2_lines = [\"**Q2 \u2014 Does DEST outperform Sobol?**\\n\"]\n        for ds in avail:\n            s_acc = _acc(all_results.get((ds, sobol),   []))\n            d_acc = _acc(all_results.get((ds, dest),    []))\n            if not s_acc or not d_acc: continue\n            cmp = Statistics.compare_groups(s_acc, d_acc)\n            q2_lines.append(\n                f\"  {ds}: DEST={np.mean(d_acc):.3f}% vs Sobol={np.mean(s_acc):.3f}% \"\n                f\"(\u0394={cmp['delta']:+.3f}pp, {_sig(cmp['p_val_ttest'])})\"\n            )\n        answers[\"Q2\"] = \"\\n\".join(q2_lines)\n\n        # \u2500\u2500 Q3 \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        q3_lines = [\"**Q3 \u2014 Does improvement grow with dataset difficulty?**\\n\"]\n        deltas = []\n        for ds in avail:\n            b_acc = _acc(all_results.get((ds, baseline), []))\n            d_acc = _acc(all_results.get((ds, dest), []))\n            if b_acc and d_acc:\n                delta = np.mean(d_acc) - np.mean(b_acc)\n                deltas.append(delta)\n                q3_lines.append(f\"  {ds}: \u0394 = {delta:+.4f}pp\")\n        if len(deltas) >= 2:\n            trend = \"INCREASING \u2705\" if deltas[-1] > deltas[0] else \"NOT clearly increasing \u274c\"\n            q3_lines.append(f\"\\n  Overall trend: {trend}\")\n        answers[\"Q3\"] = \"\\n\".join(q3_lines)\n\n        # \u2500\u2500 Q4 \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        q4_lines = [\"**Q4 \u2014 Does DEST reduce variance across seeds?**\\n\"]\n        for ds in avail:\n            b_acc = _acc(all_results.get((ds, baseline), []))\n            d_acc = _acc(all_results.get((ds, dest), []))\n            if not b_acc or not d_acc: continue\n            vr = Statistics.variance_reduction(b_acc, d_acc)\n            verdict = \"REDUCED \u2705\" if vr[\"pct_reduction\"] > 0 else \"INCREASED \u274c\"\n            q4_lines.append(\n                f\"  {ds}: \u03c3_random={np.std(b_acc):.4f}%  \u03c3_DEST={np.std(d_acc):.4f}%  \"\n                f\"Reduction={vr['pct_reduction']:.1f}%  \u2192 {verdict}\"\n            )\n        answers[\"Q4\"] = \"\\n\".join(q4_lines)\n\n        # \u2500\u2500 Q5 \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        q5_lines = [\"**Q5 \u2014 Does DEST converge faster?**\\n\"]\n        for ds in avail:\n            b_runs = all_results.get((ds, baseline), [])\n            d_runs = all_results.get((ds, dest), [])\n            b_conv = [r.convergence_epoch_90 for r in b_runs if r.convergence_epoch_90]\n            d_conv = [r.convergence_epoch_90 for r in d_runs if r.convergence_epoch_90]\n            if b_conv and d_conv:\n                delta_ep = np.mean(d_conv) - np.mean(b_conv)\n                verdict = \"FASTER \u2705\" if delta_ep < 0 else (\"SAME \u27a1\ufe0f\" if delta_ep == 0 else \"SLOWER \u274c\")\n                q5_lines.append(\n                    f\"  {ds}: Random={np.mean(b_conv):.1f} ep  DEST={np.mean(d_conv):.1f} ep  \"\n                    f\"\u0394={delta_ep:+.1f}  \u2192 {verdict}\"\n                )\n            else:\n                q5_lines.append(f\"  {ds}: convergence data insufficient.\")\n        answers[\"Q5\"] = \"\\n\".join(q5_lines)\n\n        # \u2500\u2500 Q6 \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        q6_lines = [\"**Q6 \u2014 Is the computational overhead of DEST acceptable?**\\n\"]\n        for ds in avail:\n            b_runs = all_results.get((ds, baseline), [])\n            d_runs = all_results.get((ds, dest), [])\n            if not b_runs or not d_runs: continue\n            b_t = np.mean([np.mean(r.train_time_per_epoch) for r in b_runs])\n            d_t = np.mean([np.mean(r.train_time_per_epoch) for r in d_runs])\n            b_s = np.mean([np.mean(r.sampler_time_per_epoch) for r in b_runs])\n            d_s = np.mean([np.mean(r.sampler_time_per_epoch) for r in d_runs])\n            overhead_pct = (d_s / b_s - 1) * 100 if b_s > 1e-9 else 0\n            verdict = \"ACCEPTABLE \u2705\" if overhead_pct < 10 else \"HIGH \u26a0\ufe0f\"\n            q6_lines.append(\n                f\"  {ds}: sampler_time random={b_s*1e3:.2f}ms  DEST={d_s*1e3:.2f}ms  \"\n                f\"overhead={overhead_pct:+.1f}%  \u2192 {verdict}\"\n            )\n        answers[\"Q6\"] = \"\\n\".join(q6_lines)\n\n        return answers\n\n    # \u2500\u2500 full markdown report \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    def generate_scaling_report(\n        self,\n        all_results: Dict,\n        dataset_order: List[str],\n        config: Dict,\n        stats_summary: List[Dict],\n    ) -> str:\n        mode = config.get(\"execution_mode\", \"?\")\n        n_seeds = len(config.get(\"seeds\", []))\n        avail = [d for d in dataset_order if any((d, s) in all_results for s in [\"stochastic\"])]\n\n        answers = self.answer_six_questions(all_results, avail, config)\n\n        # per-dataset results table\n        table_rows = [\"| Dataset | Method | Test Acc (%) | \u0394 vs Random | Cohen's d | p-value | Significant |\",\n                      \"|---|---|:---:|:---:|:---:|:---:|:---:|\"]\n        samplers = [\"stochastic\", \"sobol\", \"collatz_v1\", \"collatz_v2\", \"collatz_v3\"]\n        for ds in avail:\n            b_acc = _acc(all_results.get((ds, \"stochastic\"), []))\n            for s in samplers:\n                runs = all_results.get((ds, s), [])\n                if not runs: continue\n                accs = _acc(runs)\n                if s == \"stochastic\":\n                    table_rows.append(\n                        f\"| **{ds}** | **Random (baseline)** | **{np.mean(accs):.3f} \u00b1 {np.std(accs):.3f}** | \u2014 | \u2014 | \u2014 | \u2014 |\"\n                    )\n                else:\n                    cmp = Statistics.compare_groups(b_acc, accs)\n                    sig = \"**YES**\" if cmp[\"is_significant_ttest\"] else \"No\"\n                    table_rows.append(\n                        f\"| {ds} | {s} | {np.mean(accs):.3f} \u00b1 {np.std(accs):.3f} | \"\n                        f\"{cmp['delta']:+.3f}pp | {cmp['cohens_d']:.2f} | {cmp['p_val_ttest']:.4f} | {sig} |\"\n                    )\n\n        report = f\"\"\"# DEST Phase 2 \u2014 Scaling Validation Report\n\n**Mode**: {mode} | **Seeds**: {n_seeds} | **Datasets**: {', '.join(avail)}\n\n---\n\n## Summary Table\n\n{chr(10).join(table_rows)}\n\n---\n\n## Six Core Empirical Questions\n\n{answers.get('Q1', '')}\n\n---\n\n{answers.get('Q2', '')}\n\n---\n\n{answers.get('Q3', '')}\n\n---\n\n{answers.get('Q4', '')}\n\n---\n\n{answers.get('Q5', '')}\n\n---\n\n{answers.get('Q6', '')}\n\n---\n\n## Conclusion\n\nThis Phase 2 experiment {\"confirms\" if len(avail) >= 2 else \"begins to test\"} whether\nDEST generalizes beyond MNIST to harder computer vision benchmarks.\n{\"The scaling plot is the definitive visualization.\" if len(avail) >= 2 else \"\"}\n{\"Run in PAPER or FULL mode for statistically robust conclusions.\" if n_seeds < 10 else \"\"}\n\n*Auto-generated by DEST Scaling Validation framework \u2014 Phase 2*\n\"\"\"\n        report_path = os.path.join(self.reports_dir, \"scaling_report.md\")\n        with open(report_path, \"w\") as f:\n            f.write(report)\n        print(f\"\u2705 Report saved \u2192 {report_path}\")\n        return report\n\n    # \u2500\u2500 LaTeX table \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    def generate_latex_table(self, all_results: Dict, dataset_order: List[str]):\n        samplers = [\"stochastic\", \"sobol\", \"collatz_v1\", \"collatz_v2\", \"collatz_v3\"]\n        avail = [d for d in dataset_order if any((d, s) in all_results for s in samplers)]\n\n        rows = []\n        for ds in avail:\n            b_acc = _acc(all_results.get((ds, \"stochastic\"), []))\n            for s in samplers:\n                runs = all_results.get((ds, s), [])\n                if not runs: continue\n                accs = _acc(runs)\n                delta = f\"{np.mean(accs) - np.mean(b_acc):+.3f}\" if s != \"stochastic\" else \"\u2014\"\n                rows.append(\n                    f\"  {ds} & {s} & ${np.mean(accs):.3f} \\\\pm {np.std(accs):.3f}$ & {delta} \\\\\\\\\"\n                )\n\n        latex = (\"\\\\begin{table}[h]\\\\centering\\n\"\n                 \"\\\\caption{DEST Phase 2 Scaling Validation Results}\\n\"\n                 \"\\\\begin{tabular}{llcc}\\n\\\\hline\\n\"\n                 \"Dataset & Method & Test Acc (\\\\%) & $\\\\Delta$ vs Random \\\\\\\\\\n\\\\hline\\n\"\n                 + \"\\n\".join(rows) +\n                 \"\\n\\\\hline\\n\\\\end{tabular}\\n\\\\end{table}\\n\")\n\n        path = os.path.join(self.tables_dir, \"Table2_scaling_results.tex\")\n        with open(path, \"w\") as f:\n            f.write(latex)\n\n    # \u2500\u2500 legacy methods kept for backward compat \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    def generate_markdown_report(self, stats_summary: List[Dict], config: Dict) -> str:\n        mode = config.get(\"execution_mode\", \"?\")\n        dataset = config.get(\"dataset\", \"?\")\n        n_seeds = len(config.get(\"seeds\", []))\n\n        rows = []\n        for s in stats_summary:\n            sig = \"**YES**\" if s.get(\"is_significant_holm\") or s.get(\"is_significant_ttest\") else \"No\"\n            p = s.get(\"p_val_corrected\", s.get(\"p_val_ttest\", 1.0))\n            p = p if isinstance(p, float) and np.isfinite(p) else 1.0\n            rows.append(\n                f\"| **{s['mode'].upper()}** | {s['mean_acc']:.2f}% \u00b1 {s['std_acc']:.2f}% | \"\n                f\"{s['mean_loss']:.4f} | {s['cohens_d']:.2f} | {p:.4f} | {sig} |\"\n            )\n\n        report = f\"\"\"# Reporte de Investigaci\u00f3n Oficial \u2014 Proyecto DEST\n\n**Modo de Ejecuci\u00f3n**: {mode}\n**Dataset**: {dataset} | **Semillas**: {n_seeds}\n\n## 1. Resumen de Resultados Cuantitativos\n\n| M\u00e9todo | Test Accuracy (%) | Test Loss | Cohen's d | p-value (Holm) | Significativo |\n|---|:---:|:---:|:---:|:---:|:---:|\n{chr(10).join(rows)}\n\n## 2. Conclusiones Principales\n\n- Los m\u00e9todos deterministas demostraron menor variabilidad entre semillas.\n- Ver scaling_report.md para el an\u00e1lisis de Fase 2.\n\"\"\"\n        path = os.path.join(self.output_dir, \"final_report.md\")\n        with open(path, \"w\") as f:\n            f.write(report)\n        return report\n",
    "reproducibility.py": "import os\nimport random\nimport numpy as np\nimport torch\n\ndef seed_everything(seed: int) -> dict:\n    random.seed(seed)\n    os.environ['PYTHONHASHSEED'] = str(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed(seed)\n        torch.cuda.manual_seed_all(seed)\n        torch.backends.cudnn.deterministic = True\n        torch.backends.cudnn.benchmark = False\n\n    return {\n        \"seed\": seed,\n        \"torch_initial_seed\": torch.initial_seed(),\n        \"cuda_available\": torch.cuda.is_available(),\n        \"cudnn_deterministic\": torch.backends.cudnn.deterministic if torch.cuda.is_available() else None\n    }\n",
    "runner.py": "\"\"\"\nrunner.py \u2014 Core training loop with checkpoint/resume support and LR scheduling.\n\nKey changes vs v1:\n  - run_single_seed accepts a dataset name and resolves model automatically.\n  - Supports 'cosine' and 'constant' LR schedules.\n  - Checkpoint file is keyed by (dataset, sampler, seed) \u2014 robust to Colab\n    disconnects across multi-dataset runs.\n  - RunResult now carries dataset and sampler_name fields.\n  - All random seeds are re-applied at the start of each seed run.\n\"\"\"\n\nimport os\nimport time\nimport json\nimport dataclasses\nimport torch\nimport torch.nn as nn\nimport torch.optim as optim\nfrom torch.utils.data import DataLoader\nimport numpy as np\n\nfrom .config import RunResult, DATASET_MODEL_MAP\nfrom .reproducibility import seed_everything\nfrom .datasets import DatasetLoader\nfrom .samplers import SamplerFactory\nfrom .models import ModelFactory\nfrom .metrics import evaluate_model\nfrom .manifest import ExperimentManifest\n\n\nclass ExperimentRunner:\n    \"\"\"\n    Runs a single (dataset, sampler, seed) triple and saves the RunResult.\n\n    Parameters\n    ----------\n    config : dict\n        Output of dest_lib.config.get_config().  The 'dataset' key is read\n        from here; it can also be overridden via run_single_seed(dataset=...).\n    \"\"\"\n\n    def __init__(self, config: dict):\n        self.config     = config\n        self.output_dir = config[\"output_dir\"]\n        os.makedirs(self.output_dir, exist_ok=True)\n        self.manifest = ExperimentManifest(self.output_dir)\n\n        if config.get(\"device\", \"auto\") == \"auto\":\n            self.device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n        else:\n            self.device = torch.device(config[\"device\"])\n\n    # \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    def run_single_seed(\n        self,\n        exp_id: str,\n        sampler_name: str,\n        seed: int,\n        dataset: str | None = None,\n        dropout_mode: str = \"stochastic\",\n        alpha_fixed: float = 0.5,\n        # legacy compat\n        mode: str | None = None,\n    ) -> RunResult:\n        \"\"\"\n        Train one model for one seed and return the RunResult.\n\n        Parameters\n        ----------\n        dataset : str, optional\n            Override the dataset in self.config[\"dataset\"].\n        mode : str, optional\n            Legacy alias for sampler_name (ignored if sampler_name given).\n        \"\"\"\n        # -- resolve dataset & model -------------------------------------------\n        ds_name = (dataset or self.config.get(\"dataset\", \"MNIST\")).upper()\n\n        model_name = self.config.get(\"model_override\") or \\\n                     DATASET_MODEL_MAP.get(ds_name, \"SmallCNN\")\n\n        # -- reproducibility ---------------------------------------------------\n        seed_everything(seed)\n\n        # -- load data ---------------------------------------------------------\n        loader = DatasetLoader(\n            dataset_name=ds_name,\n            val_fraction=self.config.get(\"val_fraction\", 0.1),\n            split_seed=0,          # fixed split across all seeds\n            data_root=\"./data\",\n        )\n        result = loader.get_datasets()\n        # result = (train_ds, val_ds, test_ds, n_classes, input_shape, available)\n        if result[-1] is False:\n            raise RuntimeError(f\"Dataset {ds_name} is not available.\")\n        train_ds, val_ds, test_ds, n_classes, input_shape, _ = result\n\n        # -- samplers ----------------------------------------------------------\n        epochs = self.config[\"epochs\"]\n        sampler = SamplerFactory.get_sampler(\n            sampler_name=sampler_name,\n            data_source=train_ds,\n            seed=seed,\n            total_epochs=epochs,\n            alpha_fixed=alpha_fixed,\n        )\n\n        nw = self.config.get(\"num_workers\", 2)\n        train_loader = DataLoader(\n            train_ds,\n            batch_size=self.config[\"batch_size\"],\n            sampler=sampler,\n            num_workers=nw,\n            pin_memory=torch.cuda.is_available(),\n        )\n        val_loader  = DataLoader(val_ds,  batch_size=512, shuffle=False, num_workers=nw)\n        test_loader = DataLoader(test_ds, batch_size=512, shuffle=False, num_workers=nw)\n\n        # -- model & optimizer ------------------------------------------------\n        model = ModelFactory.get_model(\n            model_name=model_name,\n            input_shape=input_shape,\n            num_classes=n_classes,\n            dropout_mode=dropout_mode,\n        ).to(self.device)\n\n        opt_name = self.config.get(\"optimizer\", \"SGD\").upper()\n        if opt_name == \"SGD\":\n            optimizer = optim.SGD(\n                model.parameters(),\n                lr=self.config[\"lr\"],\n                momentum=self.config.get(\"momentum\", 0.9),\n                weight_decay=self.config.get(\"weight_decay\", 1e-4),\n            )\n        else:\n            optimizer = optim.Adam(\n                model.parameters(),\n                lr=self.config[\"lr\"],\n                weight_decay=self.config.get(\"weight_decay\", 1e-4),\n            )\n\n        # LR scheduler\n        schedule = self.config.get(\"lr_schedule\", \"constant\")\n        if schedule == \"cosine\":\n            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)\n        else:\n            scheduler = None\n\n        criterion = nn.CrossEntropyLoss()\n\n        # -- training tracking -------------------------------------------------\n        train_losses, val_losses, test_losses   = [], [], []\n        train_accs,   val_accs,   test_accs     = [], [], []\n        gen_gaps                                 = []\n        f1_epochs, prec_epochs, rec_epochs       = [], [], []\n        sampler_times, train_times, eval_times, total_times = [], [], [], []\n        samples_per_sec                          = []\n\n        start_total = time.time()\n        conv_90 = conv_95 = None\n\n        for epoch in range(epochs):\n            ep_start = time.time()\n\n            # sampler epoch update\n            t0 = time.time()\n            if hasattr(sampler, \"set_epoch\"):\n                sampler.set_epoch(epoch)\n            t_sampler = time.time() - t0\n\n            # train\n            t0 = time.time()\n            model.train()\n            run_loss = correct = total = 0\n\n            for data, target in train_loader:\n                data, target = data.to(self.device), target.to(self.device)\n                optimizer.zero_grad()\n                out  = model(data)\n                loss = criterion(out, target)\n                loss.backward()\n                optimizer.step()\n\n                run_loss += loss.item() * data.size(0)\n                correct  += (out.argmax(1) == target).sum().item()\n                total    += data.size(0)\n\n            if scheduler:\n                scheduler.step()\n\n            t_train = time.time() - t0\n            tr_loss = run_loss / max(1, total)\n            tr_acc  = correct / max(1, total) * 100.0\n\n            # eval\n            t0 = time.time()\n            v_loss, v_acc, *_ = evaluate_model(model, self.device, val_loader,  criterion)\n            te_loss, te_acc, te_f1, te_prec, te_rec, te_ece = evaluate_model(\n                model, self.device, test_loader, criterion\n            )\n            t_eval = time.time() - t0\n\n            ep_total   = time.time() - ep_start\n            throughput = total / max(1e-6, t_train)\n\n            # record\n            train_losses.append(tr_loss);  val_losses.append(v_loss);  test_losses.append(te_loss)\n            train_accs.append(tr_acc);     val_accs.append(v_acc);     test_accs.append(te_acc)\n            gen_gaps.append(tr_acc - te_acc)\n            f1_epochs.append(te_f1);  prec_epochs.append(te_prec);  rec_epochs.append(te_rec)\n            sampler_times.append(t_sampler); train_times.append(t_train)\n            eval_times.append(t_eval);       total_times.append(ep_total)\n            samples_per_sec.append(throughput)\n\n            if conv_90 is None and te_acc >= 90.0: conv_90 = epoch + 1\n            if conv_95 is None and te_acc >= 95.0: conv_95 = epoch + 1\n\n            if self.config.get(\"verbose\", True):\n                print(f\"    Epoch {epoch+1}/{epochs}  \"\n                      f\"tr_loss={tr_loss:.4f}  te_acc={te_acc:.2f}%  \"\n                      f\"lr={optimizer.param_groups[0]['lr']:.5f}\")\n\n        total_runtime = time.time() - start_total\n        gpu_mem = (torch.cuda.max_memory_allocated() / 1024 / 1024\n                   if torch.cuda.is_available() else 0.0)\n\n        res = RunResult(\n            experiment_id=exp_id,\n            dataset=ds_name,\n            sampler_name=sampler_name,\n            seed=seed,\n            mode=sampler_name,\n            train_losses=train_losses,        val_losses=val_losses,\n            test_losses=test_losses,          train_accs=train_accs,\n            val_accs=val_accs,                test_accs=test_accs,\n            generalization_gaps=gen_gaps,\n            f1_per_epoch=f1_epochs,           precision_per_epoch=prec_epochs,\n            recall_per_epoch=rec_epochs,\n            final_test_acc=test_accs[-1],     final_test_loss=test_losses[-1],\n            final_f1=f1_epochs[-1],           final_precision=prec_epochs[-1],\n            final_recall=rec_epochs[-1],      final_ece=te_ece,\n            final_generalization_gap=gen_gaps[-1],\n            convergence_epoch_90=conv_90,     convergence_epoch_95=conv_95,\n            best_test_acc=float(np.max(test_accs)),\n            best_test_epoch=int(np.argmax(test_accs)) + 1,\n            sampler_time_per_epoch=sampler_times,\n            train_time_per_epoch=train_times,\n            eval_time_per_epoch=eval_times,\n            total_time_per_epoch=total_times,\n            total_runtime_seconds=total_runtime,\n            samples_per_second=samples_per_sec,\n            gpu_memory_peak_mb=gpu_mem,\n            train_loss_variance=float(np.var(train_losses[-3:])),\n            test_acc_variance=float(np.var(test_accs[-3:])),\n            config_snapshot=self.config,\n            timestamp=time.strftime(\"%Y-%m-%dT%H:%M:%SZ\", time.gmtime()),\n        )\n\n        # -- persist -----------------------------------------------------------\n        out_file = os.path.join(\n            self.output_dir, f\"{exp_id}_{sampler_name}_seed_{seed}.json\"\n        )\n        with open(out_file, \"w\") as f:\n            json.dump(dataclasses.asdict(res), f, indent=2)\n\n        self.manifest.mark_seed_complete(exp_id, seed)\n        return res\n",
    "samplers.py": "import numpy as np\nimport torch\nfrom torch.utils.data import Sampler\n\nclass StochasticSampler(Sampler):\n    def __init__(self, data_source, seed=42):\n        super().__init__()\n        self.num_samples = len(data_source)\n        self.seed = seed\n        self.epoch = 0\n        self.generator = torch.Generator()\n        if seed is not None:\n            self.generator.manual_seed(seed)\n\n    def set_epoch(self, epoch):\n        self.epoch = epoch\n        if self.seed is not None:\n            self.generator.manual_seed(self.seed + self.epoch)\n\n    def __iter__(self):\n        indices = torch.randperm(self.num_samples, generator=self.generator).tolist()\n        return iter(indices)\n\n    def __len__(self):\n        return self.num_samples\n\nclass SequentialBaseSampler(Sampler):\n    def __init__(self, data_source):\n        super().__init__()\n        self.num_samples = len(data_source)\n    def set_epoch(self, epoch):\n        pass\n    def __iter__(self):\n        return iter(range(self.num_samples))\n    def __len__(self):\n        return self.num_samples\n\nclass SobolPermutationSampler(Sampler):\n    def __init__(self, data_source, seed=42):\n        super().__init__()\n        self.num_samples = len(data_source)\n        self.seed = seed\n        self.epoch = 0\n\n    def set_epoch(self, epoch):\n        self.epoch = epoch\n\n    def __iter__(self):\n        current_seed = self.seed + self.epoch\n        try:\n            from scipy.stats import qmc\n            sampler = qmc.Sobol(d=1, scramble=True, seed=current_seed)\n            sobol_points = sampler.random(n=self.num_samples).flatten()\n            indices = np.argsort(sobol_points).tolist()\n        except ImportError:\n            a = 1664525\n            c = 1013904223\n            m = 2**32\n            val = current_seed\n            sequence = []\n            for _ in range(self.num_samples):\n                val = (a * val + c) % m\n                sequence.append(val)\n            indices = np.argsort(sequence).tolist()\n        return iter(indices)\n\n    def __len__(self):\n        return self.num_samples\n\nclass CollatzFix1Sampler(Sampler):\n    def __init__(self, data_source, seed=42, K=50, c=1):\n        super().__init__()\n        self.num_samples = len(data_source)\n        self.seed = seed\n        self.K = K\n        self.c = c\n        self.epoch = 0\n\n    def set_epoch(self, epoch):\n        self.epoch = epoch\n\n    def _collatz_step(self, x, c):\n        val = 3 * x + c\n        v2 = (val & -val).bit_length() - 1\n        return val >> v2\n\n    def __iter__(self):\n        offset = self.seed * 10000 + self.epoch * 7919\n        values = np.empty(self.num_samples, dtype=np.float64)\n        for i in range(self.num_samples):\n            x = 2 * (i + offset) + 1\n            for _ in range(self.K):\n                x = self._collatz_step(x, self.c)\n            values[i] = x\n        indices = np.argsort(values).tolist()\n        return iter(indices)\n\n    def __len__(self):\n        return self.num_samples\n\nclass CollatzFix2Sampler(Sampler):\n    def __init__(self, data_source, seed=42, K=50, noise_ratio=0.01):\n        super().__init__()\n        self.num_samples = len(data_source)\n        self.seed = seed\n        self.K = K\n        self.noise_ratio = noise_ratio\n        self.epoch = 0\n        self.PRIMES = [3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53]\n\n    def set_epoch(self, epoch):\n        self.epoch = epoch\n\n    def _collatz_step(self, x, c):\n        val = 3 * x + c\n        v2 = (val & -val).bit_length() - 1\n        return val >> v2\n\n    def __iter__(self):\n        c = self.PRIMES[self.epoch % len(self.PRIMES)]\n        offset = self.seed * 10000 + self.epoch * 7919\n        values = np.empty(self.num_samples, dtype=np.float64)\n        for i in range(self.num_samples):\n            x = 2 * (i + offset) + 1\n            for _ in range(self.K):\n                x = self._collatz_step(x, c)\n            values[i] = x\n\n        vmin, vmax = values.min(), values.max()\n        norm_values = (values - vmin) / (vmax - vmin + 1e-10)\n        rng = np.random.RandomState(self.seed + self.epoch * 1337)\n        noise = rng.uniform(-self.noise_ratio, self.noise_ratio, self.num_samples)\n        indices = np.argsort(norm_values + noise).tolist()\n        return iter(indices)\n\n    def __len__(self):\n        return self.num_samples\n\nclass CollatzFix3Sampler(Sampler):\n    def __init__(self, data_source, total_epochs=15, alpha_start=0.0, alpha_end=0.5, seed=42, K=50):\n        super().__init__()\n        self.num_samples = len(data_source)\n        self.total_epochs = total_epochs\n        self.alpha_start = alpha_start\n        self.alpha_end = alpha_end\n        self.seed = seed\n        self.K = K\n        self.epoch = 0\n        self.PRIMES = [3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53]\n\n    def set_epoch(self, epoch):\n        self.epoch = epoch\n\n    def get_alpha(self):\n        t = self.epoch / max(1, self.total_epochs - 1)\n        return self.alpha_start + (self.alpha_end - self.alpha_start) * (1.0 - np.cos(np.pi * t)) / 2.0\n\n    def _collatz_step(self, x, c):\n        val = 3 * x + c\n        v2 = (val & -val).bit_length() - 1\n        return val >> v2\n\n    def __iter__(self):\n        alpha = self.get_alpha()\n        c = self.PRIMES[self.epoch % len(self.PRIMES)]\n        offset = self.seed * 10000 + self.epoch * 7919\n        values = np.empty(self.num_samples, dtype=np.float64)\n        for i in range(self.num_samples):\n            x = 2 * (i + offset) + 1\n            for _ in range(self.K):\n                x = self._collatz_step(x, c)\n            values[i] = x\n\n        vmin, vmax = values.min(), values.max()\n        norm_values = (values - vmin) / (vmax - vmin + 1e-10)\n        if alpha > 0:\n            rng = np.random.RandomState(self.seed + self.epoch * 9999)\n            noise = rng.uniform(0.0, 1.0, self.num_samples)\n            mixed = (1.0 - alpha) * norm_values + alpha * noise\n            indices = np.argsort(mixed).tolist()\n        else:\n            indices = np.argsort(norm_values).tolist()\n        return iter(indices)\n\n    def __len__(self):\n        return self.num_samples\n\nclass CollatzSweepSampler(Sampler):\n    \"\"\"\n    Sampler para el Experimento J (Randomness Sweep):\n    Permite fijar alpha exacto en {0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0}\n    \"\"\"\n    def __init__(self, data_source, alpha_fixed=0.5, seed=42, K=50):\n        super().__init__()\n        self.num_samples = len(data_source)\n        self.alpha_fixed = alpha_fixed\n        self.seed = seed\n        self.K = K\n        self.epoch = 0\n        self.PRIMES = [3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53]\n\n    def set_epoch(self, epoch):\n        self.epoch = epoch\n\n    def _collatz_step(self, x, c):\n        val = 3 * x + c\n        v2 = (val & -val).bit_length() - 1\n        return val >> v2\n\n    def __iter__(self):\n        alpha = self.alpha_fixed\n        c = self.PRIMES[self.epoch % len(self.PRIMES)]\n        offset = self.seed * 10000 + self.epoch * 7919\n        values = np.empty(self.num_samples, dtype=np.float64)\n        for i in range(self.num_samples):\n            x = 2 * (i + offset) + 1\n            for _ in range(self.K):\n                x = self._collatz_step(x, c)\n            values[i] = x\n\n        vmin, vmax = values.min(), values.max()\n        norm_values = (values - vmin) / (vmax - vmin + 1e-10)\n        rng = np.random.RandomState(self.seed + self.epoch * 8888)\n        noise = rng.uniform(0.0, 1.0, self.num_samples)\n        mixed = (1.0 - alpha) * norm_values + alpha * noise\n        indices = np.argsort(mixed).tolist()\n        return iter(indices)\n\n    def __len__(self):\n        return self.num_samples\n\nclass SamplerFactory:\n    @staticmethod\n    def get_sampler(sampler_name: str, data_source, seed: int = 42, total_epochs: int = 15, alpha_fixed: float = 0.5):\n        name = sampler_name.lower()\n        if name in ['stochastic', 'random']:\n            return StochasticSampler(data_source, seed=seed)\n        elif name in ['sequential']:\n            return SequentialBaseSampler(data_source)\n        elif name in ['sobol', 'deterministic']:\n            return SobolPermutationSampler(data_source, seed=seed)\n        elif name in ['collatz_v1', 'collatzv1']:\n            return CollatzFix1Sampler(data_source, seed=seed)\n        elif name in ['collatz_v2', 'collatzv2']:\n            return CollatzFix2Sampler(data_source, seed=seed)\n        elif name in ['collatz_v3', 'collatzv3', 'dest']:\n            return CollatzFix3Sampler(data_source, total_epochs=total_epochs, seed=seed)\n        elif name in ['collatz_sweep', 'sweep']:\n            return CollatzSweepSampler(data_source, alpha_fixed=alpha_fixed, seed=seed)\n        else:\n            raise ValueError(f\"Sampler desconocido: {sampler_name}\")\n",
    "statistics.py": "\"\"\"\nstatistics.py \u2014 Statistical analysis utilities for DEST experiments.\n\nChanges vs v1:\n  - NaN-safe p-value handling (occurs when comparing baseline to itself).\n  - Cleaner Holm-Bonferroni with graceful statsmodels fallback.\n  - compare_groups returns all fields even with minimal data (n=1).\n\"\"\"\n\nimport warnings\nimport numpy as np\nfrom scipy import stats\nfrom typing import Any, Dict, List, Tuple\n\n\nclass Statistics:\n\n    # \u2500\u2500 bootstrap confidence intervals \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    @staticmethod\n    def compute_ci_95_bootstrap(\n        data: List[float], n_bootstrap: int = 10_000, seed: int = 42\n    ) -> Tuple[float, float]:\n        if len(data) < 2:\n            mu = float(np.mean(data)) if data else 0.0\n            return mu, mu\n        rng   = np.random.RandomState(seed)\n        means = [np.mean(rng.choice(data, size=len(data), replace=True))\n                 for _ in range(n_bootstrap)]\n        return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))\n\n    # \u2500\u2500 Cohen's d \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    @staticmethod\n    def cohens_d(group_a: List[float], group_b: List[float]) -> float:\n        a, b = np.array(group_a), np.array(group_b)\n        if len(a) < 2 or len(b) < 2:\n            return 0.0\n        s1, s2 = np.var(a, ddof=1), np.var(b, ddof=1)\n        n1, n2 = len(a), len(b)\n        s_p = np.sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2))\n        return 0.0 if s_p == 0 else float((np.mean(a) - np.mean(b)) / s_p)\n\n    # \u2500\u2500 full group comparison \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    @staticmethod\n    def compare_groups(\n        baseline: List[float], treatment: List[float]\n    ) -> Dict[str, Any]:\n        b, t = np.array(baseline), np.array(treatment)\n\n        # -- t-test -----------------------------------------------------------\n        with warnings.catch_warnings():\n            warnings.simplefilter(\"ignore\")\n            if len(b) == len(t) and len(b) >= 2:\n                t_stat, p_ttest = stats.ttest_rel(t, b)\n            elif len(b) >= 2 and len(t) >= 2:\n                t_stat, p_ttest = stats.ttest_ind(t, b)\n            else:\n                t_stat, p_ttest = 0.0, 1.0\n\n        # Guard NaN (e.g. identical arrays)\n        p_ttest = float(p_ttest) if np.isfinite(p_ttest) else 1.0\n\n        # -- Wilcoxon / Mann-Whitney ------------------------------------------\n        with warnings.catch_warnings():\n            warnings.simplefilter(\"ignore\")\n            try:\n                if len(b) == len(t) and len(b) >= 4:\n                    w_stat, p_wil = stats.wilcoxon(t, b)\n                elif len(b) >= 4 and len(t) >= 4:\n                    w_stat, p_wil = stats.mannwhitneyu(t, b, alternative=\"two-sided\")\n                else:\n                    w_stat, p_wil = 0.0, 1.0\n            except Exception:\n                w_stat, p_wil = 0.0, 1.0\n        p_wil = float(p_wil) if np.isfinite(p_wil) else 1.0\n\n        d = Statistics.cohens_d(list(treatment), list(baseline))\n        ci_lo, ci_hi = Statistics.compute_ci_95_bootstrap(list(treatment))\n        delta = float(np.mean(t) - np.mean(b)) if len(b) else 0.0\n\n        return {\n            \"mean_treatment\":      float(np.mean(t))  if len(t) else 0.0,\n            \"std_treatment\":       float(np.std(t))   if len(t) else 0.0,\n            \"ci95_low\":            ci_lo,\n            \"ci95_high\":           ci_hi,\n            \"mean_baseline\":       float(np.mean(b))  if len(b) else 0.0,\n            \"std_baseline\":        float(np.std(b))   if len(b) else 0.0,\n            \"delta\":               delta,\n            \"p_val_ttest\":         p_ttest,\n            \"p_val_wilcoxon\":      p_wil,\n            \"cohens_d\":            float(d),\n            \"is_significant_ttest\":    bool(p_ttest < 0.05),\n            \"is_significant_wilcoxon\": bool(p_wil   < 0.05),\n        }\n\n    # \u2500\u2500 Holm-Bonferroni \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    @staticmethod\n    def apply_holm_bonferroni(\n        p_values: List[float], alpha: float = 0.05\n    ) -> Tuple[List[bool], List[float]]:\n        p = [v if np.isfinite(v) else 1.0 for v in p_values]   # sanitize NaN\n\n        try:\n            from statsmodels.stats.multitest import multipletests\n            rej, p_corr, _, _ = multipletests(p, alpha=alpha, method=\"holm\")\n            return rej.tolist(), p_corr.tolist()\n        except ImportError:\n            pass\n\n        # manual fallback\n        n = len(p)\n        order = np.argsort(p)\n        p_corr = np.ones(n)\n        rej    = np.zeros(n, dtype=bool)\n        for rank, idx in enumerate(order):\n            adj = min(1.0, p[idx] * (n - rank))\n            p_corr[idx] = adj\n            if adj < alpha:\n                rej[idx] = True\n        return rej.tolist(), p_corr.tolist()\n\n    # \u2500\u2500 variance comparison summary \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    @staticmethod\n    def variance_reduction(baseline: List[float], treatment: List[float]) -> Dict[str, float]:\n        \"\"\"Return variance ratio and percentage reduction.\"\"\"\n        var_b = float(np.var(baseline)) if len(baseline) > 1 else 0.0\n        var_t = float(np.var(treatment)) if len(treatment) > 1 else 0.0\n        ratio    = var_t / var_b if var_b > 1e-12 else 1.0\n        pct_red  = (1.0 - ratio) * 100.0\n        return {\"var_baseline\": var_b, \"var_treatment\": var_t,\n                \"ratio\": ratio, \"pct_reduction\": pct_red}\n"
}

os.makedirs('dest_lib', exist_ok=True)
for filename, content in lib_files.items():
    fpath = os.path.join('dest_lib', filename)
    with open(fpath, 'w', encoding='utf-8') as fh:
        fh.write(content)

created = sorted(f for f in os.listdir('dest_lib') if f.endswith('.py'))
print('Archivos en dest_lib:', created)
assert 'config.py' in created and 'runner.py' in created

importlib.invalidate_caches()
for mod_name in list(sys.modules.keys()):
    if 'dest_lib' in mod_name:
        del sys.modules[mod_name]

cwd = os.path.abspath('.')
if cwd not in sys.path:
    sys.path.insert(0, cwd)

from dest_lib.config import get_config
from dest_lib.runner import ExperimentRunner
print('Imports OK')

CONFIG = get_config('PAPER')
CONFIG['output_dir'] = os.path.join(cwd, 'dest_results_paper')
os.makedirs(CONFIG['output_dir'], exist_ok=True)
runner = ExperimentRunner(CONFIG)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
FALTANTES = [
    [
        "CIFAR100",
        "stochastic",
        52
    ],
    [
        "CIFAR100",
        "sobol",
        52
    ],
    [
        "CIFAR100",
        "collatz_v1",
        52
    ],
    [
        "CIFAR100",
        "collatz_v3",
        52
    ],
    [
        "CIFAR100",
        "collatz_v2",
        52
    ],
    [
        "CIFAR100",
        "stochastic",
        53
    ],
    [
        "CIFAR100",
        "sobol",
        53
    ],
    [
        "CIFAR100",
        "collatz_v1",
        53
    ],
    [
        "CIFAR100",
        "collatz_v3",
        53
    ],
    [
        "CIFAR100",
        "collatz_v2",
        53
    ],
    [
        "CIFAR100",
        "stochastic",
        54
    ],
    [
        "CIFAR100",
        "sobol",
        54
    ],
    [
        "CIFAR100",
        "collatz_v1",
        54
    ],
    [
        "CIFAR100",
        "collatz_v3",
        54
    ],
    [
        "CIFAR100",
        "collatz_v2",
        54
    ],
    [
        "CIFAR100",
        "stochastic",
        55
    ],
    [
        "CIFAR100",
        "sobol",
        55
    ],
    [
        "CIFAR100",
        "collatz_v1",
        55
    ],
    [
        "CIFAR100",
        "collatz_v3",
        55
    ],
    [
        "CIFAR100",
        "collatz_v2",
        55
    ],
    [
        "CIFAR100",
        "stochastic",
        56
    ],
    [
        "CIFAR100",
        "sobol",
        56
    ],
    [
        "CIFAR100",
        "collatz_v1",
        56
    ],
    [
        "CIFAR100",
        "collatz_v3",
        56
    ],
    [
        "CIFAR100",
        "collatz_v2",
        56
    ],
    [
        "CIFAR100",
        "stochastic",
        57
    ],
    [
        "CIFAR100",
        "sobol",
        57
    ],
    [
        "CIFAR100",
        "collatz_v1",
        57
    ],
    [
        "CIFAR100",
        "collatz_v3",
        57
    ],
    [
        "CIFAR100",
        "collatz_v2",
        57
    ],
    [
        "CIFAR100",
        "stochastic",
        58
    ],
    [
        "CIFAR100",
        "sobol",
        58
    ],
    [
        "CIFAR100",
        "collatz_v1",
        58
    ],
    [
        "CIFAR100",
        "collatz_v3",
        58
    ],
    [
        "CIFAR100",
        "collatz_v2",
        58
    ],
    [
        "CIFAR100",
        "stochastic",
        59
    ],
    [
        "CIFAR100",
        "sobol",
        59
    ],
    [
        "CIFAR100",
        "collatz_v1",
        59
    ],
    [
        "CIFAR100",
        "collatz_v3",
        59
    ],
    [
        "CIFAR100",
        "collatz_v2",
        59
    ],
    [
        "CIFAR100",
        "stochastic",
        60
    ],
    [
        "CIFAR100",
        "sobol",
        60
    ],
    [
        "CIFAR100",
        "collatz_v1",
        60
    ],
    [
        "CIFAR100",
        "collatz_v3",
        60
    ],
    [
        "CIFAR100",
        "collatz_v2",
        60
    ],
    [
        "CIFAR100",
        "stochastic",
        61
    ],
    [
        "CIFAR100",
        "sobol",
        61
    ],
    [
        "CIFAR100",
        "collatz_v1",
        61
    ],
    [
        "CIFAR100",
        "collatz_v3",
        61
    ],
    [
        "CIFAR100",
        "collatz_v2",
        61
    ],
    [
        "CIFAR100",
        "collatz_v2",
        42
    ],
    [
        "CIFAR100",
        "collatz_v2",
        43
    ],
    [
        "CIFAR100",
        "collatz_v2",
        44
    ],
    [
        "CIFAR100",
        "collatz_v2",
        45
    ],
    [
        "CIFAR100",
        "collatz_v2",
        46
    ],
    [
        "CIFAR100",
        "collatz_v2",
        47
    ],
    [
        "CIFAR100",
        "collatz_v2",
        48
    ],
    [
        "CIFAR100",
        "collatz_v2",
        49
    ],
    [
        "CIFAR100",
        "collatz_v2",
        50
    ],
    [
        "CIFAR100",
        "collatz_v2",
        51
    ]
]
METODOS_BLOQUE = ["stochastic", "sobol", "collatz_v1", "collatz_v3", "collatz_v2"]

print(f'Runs pendientes: {len(FALTANTES)} (orden pareado: semilla -> 5 metodos)')

def _path(ds, samp, seed):
    return os.path.join(CONFIG['output_dir'], f'{ds}_{samp}_{samp}_seed_{seed}.json')

# Reanudable: si Colab se cae, Celda 1 + esta otra vez; lo hecho se salta.
for idx, (ds, sampler, seed) in enumerate(FALTANTES, 1):
    if os.path.exists(_path(ds, sampler, seed)):
        print(f'[{idx}/{len(FALTANTES)}] {ds} {sampler} seed {seed}: YA EXISTE -> salto')
        continue
    print('\n' + '='*60)
    print(f'[{idx}/{len(FALTANTES)}] DS={ds} | {sampler} | SEED={seed}')
    print('='*60)
    res = runner.run_single_seed(
        exp_id=f'{ds}_{sampler}',
        sampler_name=sampler,
        seed=seed,
        dataset=ds,
        dropout_mode='deterministic',
    )
    print(f'Semilla {seed} ({sampler}) completada. Acc: {res.final_test_acc:.2f}%')

print('\n===== ANALISIS PAREADO (semillas con los 5 metodos) =====')
import glob, json, statistics as st_, math

def _cargar():
    mat = {}
    g_ = glob.glob(os.path.join(CONFIG['output_dir'], 'CIFAR100_*_seed_*.json'))
    for f_ in g_:
        with open(f_) as fh_:
            d_ = json.load(fh_)
        mat.setdefault(d_['seed'], {})[d_['sampler_name']] = d_['final_test_acc']
    return {s_: m_ for s_, m_ in sorted(mat.items()) if len(m_) == 5}

matriz = _cargar()
print(f'Semillas con quinteto completo: {sorted(matriz.keys())}')

if not matriz:
    print('Todavia no hay semillas con los 5 metodos completos.')
else:
    def _pareado(a_, b_):
        ds_ = [matriz[s_][a_] - matriz[s_][b_] for s_ in sorted(matriz)]
        n_ = len(ds_)
        md_ = st_.mean(ds_)
        sd_ = st_.stdev(ds_) if n_ > 1 else float('nan')
        t_ = md_ / (sd_ / math.sqrt(n_)) if n_ > 1 and sd_ else float('nan')
        wins_ = sum(1 for x_ in ds_ if x_ > 0)
        return ds_, md_, t_, wins_

    for base_ in ['stochastic', 'sobol']:
        ds_, md_, t_, w_ = _pareado('collatz_v3', base_)
        print(f'collatz_v3 - {base_:10s}: dMedia={md_:+.3f}pp  t_pareado={t_:+.2f}  v3_gana_{w_}/{len(ds_)}')
        print('   diffs por seed:', [f'{x_:+.2f}' for x_ in ds_])

In [ ]:
zip_filename = 'resultados_DEST_cifar100_pareado.zip'
print(f'Empaquetando en {zip_filename}...')
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(CONFIG['output_dir']):
        for file in files:
            if not file.endswith('.zip'):
                fpath = os.path.join(root, file)
                arcname = os.path.relpath(fpath, CONFIG['output_dir'])
                zipf.write(fpath, arcname)
size_mb = os.path.getsize(zip_filename) / 1024 / 1024
print(f'ZIP listo: {zip_filename} ({size_mb:.2f} MB)')
try:
    from google.colab import files
    files.download(zip_filename)
    print('Descarga iniciada.')
except ImportError:
    print(f'Descarga manual: {zip_filename}')